# Segmentation and Object Detection Studies

Portfolio source extract. See the repository README and PROVENANCE.md for methods, data requirements, original evidence and publication changes. Run from the notebooks/ directory.

In [ ]:
import sys

print("Current Python executable:")
print(sys.executable)



In [ ]:
# Install the repository requirements before running the notebook.
print("Dependencies are managed with requirements.txt.")


In [ ]:
import hashlib
import os
import random
import shutil
import subprocess
import sys
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np
import torch



In [ ]:
PROJECT_ROOT = Path("..").resolve()
DATASET_ZIP = PROJECT_ROOT / "data" / "Yolov7 Dataset.zip"
PROVIDED_WEIGHTS = PROJECT_ROOT / "data" / "yolo.pt"

YOLOV7_DIR = PROJECT_ROOT / "local_data" / "yolov7_repo"
RAW_DATA_DIR = PROJECT_ROOT / "local_data" / "yolov7_wheat_raw"
FLAT_DATASET_DIR = RAW_DATA_DIR / "Wheat Dataset"
DATASET_DIR = YOLOV7_DIR / "dataset"
REPORT_DIR = PROJECT_ROOT / "runs" / "yolov7_assignment_ab_test_200" / "report_artifacts"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

IMG_SIZE = 640
BATCH_SIZE = 16
EPOCHS = 200
WORKERS = min(8, max((os.cpu_count() or 2) - 2, 2))
DEVICE_ARG = "0" if torch.cuda.is_available() else "cpu"

TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

CLASS_NAMES = ["disease", "pest", "wheat"]
BASE_RUN_NAME = "wheat_assignment_abtest_640_bs16_e200_v1"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset zip: {DATASET_ZIP}")
print(f"Provided weights: {PROVIDED_WEIGHTS}")
print(f"img={IMG_SIZE}, batch={BATCH_SIZE}, epochs={EPOCHS}, workers={WORKERS}")
print(f"split={TRAIN_RATIO:.0%}/{VAL_RATIO:.0%}/{TEST_RATIO:.0%}")
print(f"report dir={REPORT_DIR}")



In [ ]:
if not PROVIDED_WEIGHTS.exists():
    raise FileNotFoundError(f"Weight file not found: {PROVIDED_WEIGHTS}")
if not DATASET_ZIP.exists():
    raise FileNotFoundError(f"Dataset zip not found: {DATASET_ZIP}")

if not YOLOV7_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/WongKinYiu/yolov7.git", str(YOLOV7_DIR)],
        check=True,
    )

if not FLAT_DATASET_DIR.exists():
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP, "r") as zip_ref:
        zip_ref.extractall(RAW_DATA_DIR)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(YOLOV7_DIR / "requirements.txt")],
    check=True,
)

IMAGE_DIR = FLAT_DATASET_DIR / "images"
LABEL_DIR = FLAT_DATASET_DIR / "labels"

image_paths = sorted([p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
label_paths = {p.stem: p for p in LABEL_DIR.iterdir() if p.suffix.lower() == ".txt"}
image_label_pairs = [(img, label_paths[img.stem]) for img in image_paths if img.stem in label_paths]

def load_label_rows(label_path):
    rows = []
    text = label_path.read_text().strip()
    if not text:
        return rows
    for line in text.splitlines():
        parts = line.split()
        rows.append((int(float(parts[0])), *map(float, parts[1:])))
    return rows


def image_hash(path):
    return hashlib.md5(path.read_bytes()).hexdigest()


def save_label_rows(label_path, rows):
    lines = []
    for cls_id, x, y, bw, bh in rows:
        lines.append(f"{cls_id} {x:.6f} {y:.6f} {bw:.6f} {bh:.6f}")
    label_path.write_text("\n".join(lines) + ("\n" if lines else ""))


def clip01(value):
    return min(1.0, max(0.0, value))


def sanitize_row(row):
    cls_id, x, y, bw, bh = row
    bw = clip01(bw)
    bh = clip01(bh)
    if bw <= 0.0 or bh <= 0.0:
        return None
    x = clip01(x)
    y = clip01(y)
    bw = min(bw, 2.0 * min(x, 1.0 - x))
    bh = min(bh, 2.0 * min(y, 1.0 - y))
    if bw <= 0.0 or bh <= 0.0:
        return None
    return (cls_id, x, y, bw, bh)


def cleaned_rows(label_path):
    rows = []
    for row in load_label_rows(label_path):
        cleaned_row = sanitize_row(row)
        if cleaned_row is not None:
            rows.append(cleaned_row)
    rows.sort(key=lambda row: (row[0], row[1], row[2], row[3], row[4]))
    return rows


def select_canonical_group_sample(group):
    candidates = []
    for image_path, label_path in group:
        rows = cleaned_rows(label_path)
        class_counts = Counter(row[0] for row in rows)
        candidates.append(
            {
                "image_path": image_path,
                "label_path": label_path,
                "rows": rows,
                "row_count": len(rows),
                "pest_count": class_counts[1],
                "disease_count": class_counts[0],
                "wheat_count": class_counts[2],
            }
        )

    candidates.sort(
        key=lambda candidate: (
            -candidate["row_count"],
            -candidate["pest_count"],
            -candidate["disease_count"],
            -candidate["wheat_count"],
            candidate["image_path"].name,
        )
    )
    return candidates[0]


grouped_pairs = defaultdict(list)
for image_path, label_path in image_label_pairs:
    grouped_pairs[image_hash(image_path)].append((image_path, label_path))

group_records = []
conflicting_duplicate_groups = 0
collapsed_duplicate_groups = 0
for group in grouped_pairs.values():
    sorted_group = sorted(group, key=lambda pair: pair[0].name)
    label_hashes = {hashlib.md5(label_path.read_bytes()).hexdigest() for _, label_path in sorted_group}
    if len(sorted_group) > 1:
        collapsed_duplicate_groups += 1
    if len(label_hashes) > 1:
        conflicting_duplicate_groups += 1
    canonical_sample = select_canonical_group_sample(sorted_group)
    group_records.append(
        {
            "image_path": canonical_sample["image_path"],
            "rows": canonical_sample["rows"],
            "group_size": len(sorted_group),
            "label_versions": len(label_hashes),
            "selected_label_path": canonical_sample["label_path"],
        }
    )


def group_has_pest(record):
    return any(row[0] == 1 for row in record["rows"])


pest_groups = [record for record in group_records if group_has_pest(record)]
non_pest_groups = [record for record in group_records if not group_has_pest(record)]

random.shuffle(pest_groups)
random.shuffle(non_pest_groups)

def split_groups(groups):
    total = len(groups)
    train_end = int(total * TRAIN_RATIO)
    val_end = int(total * (TRAIN_RATIO + VAL_RATIO))
    return {
        "train": groups[:train_end],
        "val": groups[train_end:val_end],
        "test": groups[val_end:],
    }


pest_split = split_groups(pest_groups)
non_pest_split = split_groups(non_pest_groups)
split_groups_map = {
    split_name: pest_split[split_name] + non_pest_split[split_name]
    for split_name in ("train", "val", "test")
}
for split_name in split_groups_map:
    random.shuffle(split_groups_map[split_name])

for split_name in ("train", "val", "test"):
    image_split_dir = DATASET_DIR / "images" / split_name
    label_split_dir = DATASET_DIR / "labels" / split_name
    if image_split_dir.exists():
        shutil.rmtree(image_split_dir)
    if label_split_dir.exists():
        shutil.rmtree(label_split_dir)
    image_split_dir.mkdir(parents=True, exist_ok=True)
    label_split_dir.mkdir(parents=True, exist_ok=True)

    for record in split_groups_map[split_name]:
        image_path = record["image_path"]
        label_path = label_split_dir / f"{image_path.stem}.txt"
        shutil.copy2(image_path, image_split_dir / image_path.name)
        save_label_rows(label_path, record["rows"])

for cache_path in (DATASET_DIR / "labels").rglob("*.cache*"):
    cache_path.unlink()

def class_counts(records):
    counts = Counter()
    for record in records:
        for class_id, *_ in record["rows"]:
            counts[CLASS_NAMES[class_id]] += 1
    return counts


for split_name in ("train", "val", "test"):
    print(f"{split_name}: {len(split_groups_map[split_name])} images")
    print(class_counts(split_groups_map[split_name]))

print(f"Unique source images: {len(group_records)}")
print(f"Duplicate image groups collapsed: {collapsed_duplicate_groups}")
print(f"Conflicting duplicate groups resolved: {conflicting_duplicate_groups}")
print(f"Dataset written to: {DATASET_DIR}")



In [ ]:
DATA_YAML_PATH = YOLOV7_DIR / "data" / "wheat.yaml"
DATA_YAML_TEXT = """train: dataset/images/train
val: dataset/images/val
test: dataset/images/test
nc: 3
names:
  0: disease
  1: pest
  2: wheat
"""

DATA_YAML_PATH.write_text(DATA_YAML_TEXT)
print(DATA_YAML_PATH.read_text())



In [ ]:
PROVIDED_TAG = "provided"
OFFICIAL_TAG = "official"

OFFICIAL_WEIGHTS = YOLOV7_DIR / "yolov7.pt"
DOWNLOAD_OFFICIAL_WEIGHTS = True

if DOWNLOAD_OFFICIAL_WEIGHTS and not OFFICIAL_WEIGHTS.exists():
    subprocess.run(
        [
            "wget",
            "https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7.pt",
            "-O",
            str(OFFICIAL_WEIGHTS),
        ],
        cwd=YOLOV7_DIR,
        check=True,
    )

EXPERIMENT_WEIGHTS = {
    PROVIDED_TAG: PROVIDED_WEIGHTS,
    OFFICIAL_TAG: OFFICIAL_WEIGHTS,
}

for experiment_tag, weights_path in EXPERIMENT_WEIGHTS.items():
    if not weights_path.exists():
        raise FileNotFoundError(f"Missing weights for {experiment_tag}: {weights_path}")
    print(f"{experiment_tag}: {weights_path}")



In [ ]:
def run_command(cmd, cwd):
    print("$", " ".join(str(x) for x in cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


def latest_run_dir(parent_dir, prefix):
    candidates = sorted(parent_dir.glob(f"{prefix}*"), key=lambda path: path.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError(f"No run found for prefix: {prefix}")
    return candidates[-1]


def run_experiment(experiment_tag, weights_path):
    train_run = f"{BASE_RUN_NAME}_{experiment_tag}_train"
    test_run = f"{BASE_RUN_NAME}_{experiment_tag}_test"
    detect_run = f"{BASE_RUN_NAME}_{experiment_tag}_detect"

    train_cmd = [
        sys.executable,
        "train.py",
        "--data",
        "data/wheat.yaml",
        "--weights",
        str(weights_path),
        "--epochs",
        str(EPOCHS),
        "--batch-size",
        str(BATCH_SIZE),
        "--img",
        str(IMG_SIZE),
        "--device",
        DEVICE_ARG,
        "--workers",
        str(WORKERS),
        "--project",
        "runs/train",
        "--name",
        train_run,
    ]
    run_command(train_cmd, cwd=YOLOV7_DIR)

    train_dir = latest_run_dir(YOLOV7_DIR / "runs" / "train", train_run)
    final_weights = train_dir / "weights" / "best.pt"
    print(f"{experiment_tag} best weights: {final_weights}")

    test_cmd = [
        sys.executable,
        "test.py",
        "--weights",
        str(final_weights),
        "--data",
        "data/wheat.yaml",
        "--task",
        "test",
        "--batch-size",
        str(BATCH_SIZE),
        "--img",
        str(IMG_SIZE),
        "--device",
        DEVICE_ARG,
        "--project",
        "runs/test",
        "--name",
        test_run,
    ]
    run_command(test_cmd, cwd=YOLOV7_DIR)
    test_dir = latest_run_dir(YOLOV7_DIR / "runs" / "test", test_run)

    detect_cmd = [
        sys.executable,
        "detect.py",
        "--weights",
        str(final_weights),
        "--source",
        "dataset/images/test",
        "--img-size",
        str(IMG_SIZE),
        "--device",
        DEVICE_ARG,
        "--project",
        "runs/detect",
        "--name",
        detect_run,
        "--save-txt",
        "--save-conf",
    ]
    run_command(detect_cmd, cwd=YOLOV7_DIR)
    detect_dir = latest_run_dir(YOLOV7_DIR / "runs" / "detect", detect_run)

    result = {
        "weights_path": weights_path,
        "train_dir": train_dir,
        "best_weights": final_weights,
        "test_dir": test_dir,
        "detect_dir": detect_dir,
    }
    print(
        experiment_tag,
        "| train:", result["train_dir"],
        "| test:", result["test_dir"],
        "| detect:", result["detect_dir"],
    )
    return result


RUN_RESULTS = {}



In [ ]:
RUN_RESULTS[PROVIDED_TAG] = run_experiment(PROVIDED_TAG, EXPERIMENT_WEIGHTS[PROVIDED_TAG])



In [ ]:
RUN_RESULTS[OFFICIAL_TAG] = run_experiment(OFFICIAL_TAG, EXPERIMENT_WEIGHTS[OFFICIAL_TAG])



In [ ]:
print("Completed experiments:")
for experiment_tag, result in RUN_RESULTS.items():
    print(
        experiment_tag,
        "| train:", result["train_dir"],
        "| test:", result["test_dir"],
        "| detect:", result["detect_dir"],
    )



In [ ]:
REPORT_EXPERIMENT_KEY = PROVIDED_TAG

if REPORT_EXPERIMENT_KEY not in RUN_RESULTS:
    raise KeyError(f"Unknown experiment key: {REPORT_EXPERIMENT_KEY}")

SELECTED_DETECT_DIR = RUN_RESULTS[REPORT_EXPERIMENT_KEY]["detect_dir"]
print(f"Using experiment '{REPORT_EXPERIMENT_KEY}' for report images: {SELECTED_DETECT_DIR}")


def load_yolo_labels(label_path):
    rows = []
    text = label_path.read_text().strip()
    if not text:
        return rows
    for line in text.splitlines():
        parts = line.split()
        rows.append((int(float(parts[0])), *map(float, parts[1:])))
    return rows


def yolo_to_xyxy_int(width, height, x_center, y_center, box_w, box_h):
    x1 = int((x_center - box_w / 2.0) * width)
    y1 = int((y_center - box_h / 2.0) * height)
    x2 = int((x_center + box_w / 2.0) * width)
    y2 = int((y_center + box_h / 2.0) * height)
    return x1, y1, x2, y2


def draw_gt(image_bgr, label_path):
    canvas = image_bgr.copy()
    height, width = canvas.shape[:2]
    for cls_id, x, y, bw, bh in load_yolo_labels(label_path):
        x1, y1, x2, y2 = yolo_to_xyxy_int(width, height, x, y, bw, bh)
        cv2.rectangle(canvas, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            canvas,
            CLASS_NAMES[int(cls_id)],
            (x1, max(20, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2,
        )
    return canvas


def add_title(image_bgr, title):
    canvas = image_bgr.copy()
    cv2.rectangle(canvas, (0, 0), (canvas.shape[1], 38), (0, 0, 0), -1)
    cv2.putText(canvas, title, (12, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    return canvas


side_by_side_dir = REPORT_DIR / f"assignment_side_by_side_{REPORT_EXPERIMENT_KEY}"
side_by_side_dir.mkdir(parents=True, exist_ok=True)

test_images = sorted((DATASET_DIR / "images" / "test").glob("*"))[:5]
for image_path in test_images:
    image_bgr = cv2.imread(str(image_path))
    gt_label = DATASET_DIR / "labels" / "test" / f"{image_path.stem}.txt"
    gt_panel = add_title(draw_gt(image_bgr, gt_label), "Ground Truth")

    pred_panel = cv2.imread(str(SELECTED_DETECT_DIR / image_path.name))
    if pred_panel is None:
        pred_panel = image_bgr.copy()
    pred_panel = cv2.resize(pred_panel, (gt_panel.shape[1], gt_panel.shape[0]))
    pred_panel = add_title(pred_panel, "Prediction")

    combined = np.concatenate([gt_panel, pred_panel], axis=1)
    save_path = side_by_side_dir / f"{image_path.stem}_gt_vs_pred.png"
    cv2.imwrite(str(save_path), combined)

print("Saved 5 side-by-side images to:", side_by_side_dir)

